# Inside the embedding space: where does the student lose the teacher?

The students are scored with one number per window, the cosine with the teacher's vector. That number compresses 512
values into one, so it can't say *what* the student gets right or *where* it fails. This notebook opens it up, with
a picture for every idea:

- **A.** What an embedding is, and why all of them lean the same way (the "cone")
- **1.** How much of the cosine comes for free (raw vs centred cosine)
- **2.** How many directions the teacher really uses (PCA), and a 2D map of teacher vs student
- **3.** Coarse or fine: which of the teacher's directions the student reproduces (R² per component)
- **4.** Relations between tracks: similarity matrices, and what the neighbours need
- **5.** **Error analysis: where the student misses, and what is special about those windows**
- **6.** One standard number (CKA) and a summary

No training: it loads trained students (`.research_cache/distill/trained/`, made by `train.py`) and predicts the test
windows **on the CPU**, so a training run can keep the GPU. A few minutes. Kernel: the research venv (`venv-distill`).
Track names appear in some outputs: they stay on this machine (the notebook is committed without outputs).

**Background reading:** Kornblith et al. 2019, *Similarity of Neural Network Representations Revisited* (CKA);
Raghu et al. 2017, *SVCCA*; Ethayarajh 2019, *How Contextual are Contextualized Word Representations?* (embeddings
in a narrow cone: anisotropy); Mu & Viswanath 2018, *All-but-the-Top* (removing the mean direction); Andrew Ng,
*Machine Learning Yearning*, chapters on error analysis.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

HERE = Path.cwd()
if not (HERE / 'common.py').exists():   # opened from the repo root instead of this folder
    HERE = HERE / 'research' / 'experiments' / 'distill'
sys.path.insert(0, str(HERE))
from common import DATA, WORK  # noqa: E402
from evaluate import load_split, positive_tags, top_k_overlap, track_vectors  # noqa: E402
from students import STUDENTS  # noqa: E402

torch.set_num_threads(4)   # CPU only: leaves the GPU to a training run

# The students to look at: run names (folders in .research_cache/distill/trained/). Add more to compare.
RUNS = {
    '1,199 tracks (baseline)': 'cnn4-small-64-n1199',
    '50 tracks (overfit test)': 'cnn4-small-64-n50-overfit',
}
BASE = '1,199 tracks (baseline)'   # the student the single-model pictures show

# Some genres are tagged two ways in the library (this notebook's own map showed it: two colours for one genre).
TAG_SYNONYMS = {'dnb': 'drum & bass', 'd&b': 'drum & bass', 'drum and bass': 'drum & bass', 'breaks': 'breakbeat'}

# Colours: categorical slots in a fixed order (only the first three in scatter plots), one-hue blue for magnitude,
# blue-gray-red for signed values. Text stays in text colours.
SERIES = ['#257de0', '#f47230', '#00b894', '#f2a600', '#ee7fac', '#008a17', '#6a4ce2', '#ec4647']
TEXT, MUTED, GRID, OTHER = '#191919', '#666666', '#e6e6e6', '#b8b8b8'
SEQUENTIAL = LinearSegmentedColormap.from_list('blue', ['#cce3fd', '#84b9f3', '#338ced', '#1b64be', '#0e3d7e'])
DIVERGING = LinearSegmentedColormap.from_list('blue-gray-red', ['#257de0', '#efefef', '#ec4647'])
plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'font.size': 10, 'axes.titlesize': 11, 'axes.titlecolor': TEXT,
    'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'lines.linewidth': 2, 'figure.dpi': 110,
    'legend.frameon': False, 'legend.labelcolor': TEXT,
})
MODEL_COLOUR = dict(zip(RUNS, SERIES))   # a model keeps its colour in every chart


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), 1e-9)

## Data

The teacher's vectors for the **training** windows are only used to fit things (the mean direction, the PCA); all
measurements are on the **test** windows. Fitting a transform on the data you measure on is a small leak: the test
set would shape its own yardstick. Same rule as for any preprocessing (scalers, PCA, vocabularies).

All vectors are made unit length, because the app compares directions only.

In [ ]:
teacher_all = np.load(DATA / 'teacher.npy', mmap_mode='r')
train_rows, _, _ = load_split('train')
test_rows, test_tracks, test_manifest = load_split('test')

T_train = unit(np.asarray(teacher_all[train_rows], dtype=np.float32))   # fits the mean and the PCA
T = unit(np.asarray(teacher_all[test_rows], dtype=np.float32))          # what we measure on
mu = T_train.mean(0)                                                      # the shared direction (not unit length)
window_index = np.load(DATA / 'windows.npz')['window'][test_rows]         # which 10 s window of its track
print(f'train: {len(T_train):,} windows; test: {len(T):,} windows of {len(np.unique(test_tracks)):,} tracks')

## Student predictions (CPU)

Each student turns the cached log-mel of a test window into a 512-d vector. The mean baseline predicts the same
vector, the training mean, for every window. While the log-mels are in memory, each window's average log-mel level
is kept too (how loud the window is), for the error analysis.

**Built-in check:** the printed window cosines must equal `test_window_cos_mean` of the same runs in MLflow. If they
don't, the notebook is not measuring what training measured.

In [ ]:
def predict_cpu(run, mel, batch=256):
    model = STUDENTS[run.rsplit('-n', 1)[0]]()
    state = torch.load(WORK / 'trained' / run / 'state_dict.pt', weights_only=True, map_location='cpu')
    model.load_state_dict(state)
    model.eval()
    out = []
    with torch.inference_mode():
        for start in range(0, len(mel), batch):
            out.append(model.embed_mel(torch.from_numpy(mel[start:start + batch]).float()).numpy())
    return unit(np.concatenate(out))


mel = np.asarray(np.load(DATA / 'mel64.npy', mmap_mode='r')[test_rows])   # about 1 GB of float16
loudness = mel.mean(axis=(1, 2), dtype=np.float32)                         # average log-mel level per window
S = {'mean vector (baseline)': np.tile(unit(mu), (len(T), 1))}
for label, run in RUNS.items():
    S[label] = predict_cpu(run, mel)
del mel
for label, s in S.items():
    print(f'{label:28s} window cosine {np.mean((s * T).sum(1)):.4f}')

## A. What an embedding is

An embedding is just a list of numbers, here 512 per track, placed so that tracks that sound alike get similar
lists. Below, three test tracks, each as the teacher's row and the student's row: blue = negative, red = positive.
Only the first 64 of the 512 numbers are drawn, so the cells are wide enough to compare by eye.

Three things to notice:
- **The numbers are small.** A unit-length vector with 512 entries has entries of about ±0.04 on average
  (1/√512), so the colour scale runs to about ±0.1, not ±1. Its length is 1; no single entry gets near 1.
- **No single column means anything** ("dimension 137 = bass" is not how it works). Only the whole row's
  *direction* matters, compared with other rows. That is why cosine similarity is the measure.
- The student's row should look like the teacher's row *of the same track*. The third row per track is the
  **difference** (student minus teacher) on the same scale: the paler it is, the closer the student. Compare it
  with the printed cosine of each track.

In [ ]:
t_tracks, track_ids = track_vectors(T, test_tracks)          # one unit vector per test track (mean of its windows)
s_tracks = {label: track_vectors(s, test_tracks)[0] for label, s in S.items()}

pick = np.random.default_rng(1).choice(len(t_tracks), 3, replace=False)
rows, names = [], []
for n, i in enumerate(pick, 1):
    rows += [t_tracks[i], s_tracks[BASE][i], s_tracks[BASE][i] - t_tracks[i]]
    names += [f'track {n}  teacher', f'track {n}  student', f'track {n}  difference']
limit = np.percentile(np.abs(t_tracks), 99)
shown = 64
print(f'a unit vector of 512 equal numbers would have each = {1 / np.sqrt(512):.3f}; '
      f'the colour scale runs to ±{limit:.3f} (99th percentile of |value|), not ±1')

fig, ax = plt.subplots(figsize=(10, 3.8))
image = ax.imshow(np.array(rows)[:, :shown], aspect='auto', cmap=DIVERGING, vmin=-limit, vmax=limit,
                  interpolation='nearest', extent=(0.5, shown + 0.5, len(rows) - 0.5, -0.5))
for n in (1, 2):
    ax.axhline(3 * n - 0.5, color='white', linewidth=4)        # a gap between tracks
ax.set_yticks(range(len(names)), names)
ax.set_xlabel(f'dimension (the first {shown} of 512)')
ax.grid(False)
ax.set_title('Three tracks as lists of numbers: teacher row vs student row')
fig.colorbar(image, ax=ax, label='value', fraction=0.025, pad=0.01)
plt.show()

for n, i in enumerate(pick, 1):
    print(f'track {n}: cosine teacher vs student {t_tracks[i] @ s_tracks[BASE][i]:.3f}')
print(f'track 1 vs track 2, both teacher: {t_tracks[pick[0]] @ t_tracks[pick[1]]:.3f}  (two different songs!)')

### The cone: all vectors lean the same way

The last printed number is the surprise: two *different* songs still have a high cosine. The teacher's vectors don't
spread over the whole sphere; they sit in a narrow cone around one shared direction (this is called
**anisotropy**, common in learned embeddings).

Left: each dot is a test window. Horizontal = how much it points along the shared direction; vertical = along the
main direction in which windows *differ*. The grey circle is where unit vectors pointing only in these two directions
would lie. Every window is far to the right: they all lean the same way.
Right: the same windows after subtracting the mean (**centring**). Now the cloud sits around the origin, and what is
left is exactly what tells tracks apart.

In [ ]:
direction = unit(mu)
_, singular, Vt = np.linalg.svd(T_train - mu, full_matrices=False)   # rows of Vt: the teacher's principal directions
sample = np.random.default_rng(2).choice(len(T), 2000, replace=False)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.6))

angle = np.linspace(0, 2 * np.pi, 300)
left.plot(np.cos(angle), np.sin(angle), color=OTHER, linewidth=1)
left.scatter(T[sample] @ direction, (T[sample] - mu) @ Vt[0], s=7, alpha=0.35, color=SERIES[0], linewidths=0)
left.annotate('', xy=(1, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=TEXT, lw=1.5), zorder=4)
left.annotate('shared direction', xy=(0.25, 0), xytext=(-0.55, 0.75), color=TEXT,
              arrowprops=dict(arrowstyle='-', color=MUTED, lw=0.8))
left.plot(0, 0, 'o', color=TEXT, ms=5, zorder=4)
left.text(-0.05, -0.15, 'origin', color=MUTED, ha='right')
left.set_aspect('equal')
left.set_xlim(-1.15, 1.15)
left.set_ylim(-1.15, 1.15)
left.set_xlabel('along the shared direction')
left.set_ylabel('along the main difference (PC 1)')
left.set_title('Raw: every window leans the same way')

centred_sample = (T[sample] - mu) @ Vt[:2].T
right.scatter(centred_sample[:, 0], centred_sample[:, 1], s=7, alpha=0.35, color=SERIES[0], linewidths=0)
right.axhline(0, color=MUTED, linewidth=0.8)
right.axvline(0, color=MUTED, linewidth=0.8)
right.set_aspect('equal')
right.set_xlabel('PC 1 (main difference)')
right.set_ylabel('PC 2 (second difference)')
right.set_title('Centred: only the differences are left')
plt.show()

## 1. How much of the cosine is free?

Because of the cone, the raw cosine starts high for any vector that points "the usual way". The **centred cosine**
(subtract the training mean from both vectors first) measures only the part that tells tracks apart, which is what
neighbours and search depend on. The mean baseline drops to about 0 there, as it should: it knows nothing about any
particular track.

In [ ]:
pairs = T[np.random.default_rng(0).choice(len(T), (2000, 2))]
print(f'cosine between two random test windows: {np.mean((pairs[:, 0] * pairs[:, 1]).sum(1)):.3f}')
print(f'cosine of a test window with the shared direction: {np.mean(T @ direction):.3f}')

raw = {label: (s * T).sum(1) for label, s in S.items()}
centred = {label: (unit(s - mu) * unit(T - mu)).sum(1) for label, s in S.items()}
display(pd.DataFrame({'raw cosine': {k: v.mean() for k, v in raw.items()},
                      'centred cosine': {k: v.mean() for k, v in centred.items()},
                      'centred, 5th percentile': {k: np.percentile(v, 5) for k, v in centred.items()}}).round(3))

fig, ax = plt.subplots(figsize=(7.5, 3.2))
bins = np.linspace(-0.4, 1, 71)
ax.hist(raw[BASE], bins=bins, histtype='step', linewidth=2, color=OTHER, label='raw cosine')
ax.hist(centred[BASE], bins=bins, histtype='step', linewidth=2, color=MODEL_COLOUR[BASE], label='centred cosine')
ax.set_title(f'Agreement per test window: {BASE}')
ax.set_xlabel('cosine with the teacher')
ax.set_ylabel('windows')
ax.legend(loc='upper left')
plt.show()

## 2. How many directions does the teacher use?

PCA finds the directions along which the teacher's (centred) training vectors vary most, sorted by how much variance
each one carries. If a few directions carry most of the variance, the space is effectively low-dimensional. The
**participation ratio**, `(sum of variances)² / sum of squared variances`, turns this into one number: the effective
count of directions (512 if all were equal, 1 if one dominated).

In [ ]:
variance = singular ** 2 / (len(T_train) - 1)
cumulative = np.cumsum(variance) / variance.sum()
for target in (0.5, 0.8, 0.9, 0.95):
    print(f'{target:.0%} of the variance: {np.searchsorted(cumulative, target) + 1} components')
print(f'effective dimensionality (participation ratio): {variance.sum() ** 2 / (variance ** 2).sum():.1f} of 512')

fig, ax = plt.subplots(figsize=(7.5, 3))
ax.plot(np.arange(1, 513), cumulative, color=SERIES[0])
ax.set_xscale('log')
ax.set_xlabel('number of principal components')
ax.set_ylabel('share of the variance')
ax.set_title("How many directions does the teacher's space use?")
plt.show()

### A map: the teacher's two main axes, teacher vs student

Each filled dot is a test track where the teacher puts it, coloured by its first genre tag (the three most common;
the rest grey; `dnb` and `drum & bass` merged: a first version of this map showed the same genre in two colours, a
small data-quality catch). The hollow ring of the same colour is where the **student** puts the same track, joined by a line.
Short lines = the student places the track where the teacher does, *on these two axes*. These two axes are only the
coarsest part of the space (see the share in the title); sections 3 and 4 look at the rest.

In [ ]:
first_tag = [next(iter(positive_tags(genre)), '') for genre in test_manifest.loc[track_ids, 'genre']]
first_tag = [TAG_SYNONYMS.get(tag, tag) for tag in first_tag]
top_tags = [tag for tag, _ in Counter(tag for tag in first_tag if tag).most_common(3)]
colour_of = {tag: SERIES[i] for i, tag in enumerate(top_tags)}
colours = np.array([colour_of.get(tag, OTHER) for tag in first_tag])

zt = (t_tracks - mu) @ Vt[:2].T
zs = (s_tracks[BASE] - mu) @ Vt[:2].T
show = np.random.default_rng(3).choice(len(zt), 300, replace=False)
show = show[np.argsort(colours[show] != OTHER)]          # grey first, so the coloured tracks are drawn on top

fig, ax = plt.subplots(figsize=(8, 6))
for i in show:
    ax.plot([zt[i, 0], zs[i, 0]], [zt[i, 1], zs[i, 1]], color=OTHER, linewidth=0.8, alpha=0.7, zorder=1)
ax.scatter(zt[show, 0], zt[show, 1], s=36, c=colours[show], edgecolors='white', linewidths=0.8, zorder=3)
ax.scatter(zs[show, 0], zs[show, 1], s=26, facecolors='none', edgecolors=colours[show], linewidths=1.2, zorder=2)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title(f'Teacher (filled) vs student (ring), 300 test tracks; these two axes: {cumulative[1]:.0%} of the variance')
handles = [Line2D([], [], marker='o', linestyle='', color=colour_of[tag], label=tag, ms=8) for tag in top_tags]
handles += [Line2D([], [], marker='o', linestyle='', color=OTHER, label='other genres', ms=8),
            Line2D([], [], marker='o', linestyle='', color=MUTED, label='teacher', ms=8),
            Line2D([], [], marker='o', linestyle='', markerfacecolor='none', color=MUTED, label='student', ms=8)]
ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1))
plt.show()

## 3. Coarse or fine: which of the teacher's directions does the student reproduce?

Both the teacher's and the student's test vectors are expressed in the teacher's principal coordinates. The first
components carry the broad differences (in music: roughly style, energy, instrumentation); the later ones the fine
detail that separates similar tracks. For each component, two separate questions:

- **Is the information there?** The correlation between the teacher's and the student's coordinate, squared (ρ²):
  1 = the student orders the windows along this direction exactly like the teacher, 0 = unrelated. It ignores
  scale and offset.
- **Is it the right size?** The student's spread along the component divided by the teacher's: 1 = same size,
  below 1 = **shrunk**. Regression models shrink where they are unsure (predicting closer to the average is the
  safe bet under a squared or cosine loss).

**R²** (how much of the teacher's variation the student reproduces) punishes both at once, which is why it is in the
table, summed per block with each component weighted by its variance. A plain mean of per-component R² would explode
on the last components, where the teacher barely varies and any small error divided by almost nothing is huge.

In [ ]:
Zt = (T - mu) @ Vt.T                       # the teacher's test windows in its own principal coordinates
spread = ((Zt - Zt.mean(0)) ** 2).sum(0)   # the teacher's variation per component
Zs = {label: (S[label] - mu) @ Vt.T for label in RUNS}
rho2 = {label: np.array([np.corrcoef(Zt[:, k], z[:, k])[0, 1] ** 2 for k in range(512)]) for label, z in Zs.items()}
size = {label: z.std(0) / Zt.std(0) for label, z in Zs.items()}

x = np.arange(1, 513)
fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for label in RUNS:
    for ax, values in ((top, rho2[label]), (bottom, size[label])):
        ax.plot(x, values, color=MODEL_COLOUR[label], linewidth=0.8, alpha=0.3)
        ax.plot(x, pd.Series(values).rolling(9, center=True, min_periods=1).mean(), color=MODEL_COLOUR[label],
                label=label)
top.set_ylim(0, 1)
top.set_ylabel('ρ² (information there)')
top.set_title("Which of the teacher's directions does the student reproduce?")
top.legend(loc='upper right')
bottom.axhline(1, color=MUTED, linewidth=0.8, linestyle='--')
bottom.text(1.1, 1.03, 'same size as the teacher', color=MUTED, fontsize=9, va='bottom')
bottom.set_ylim(0, 2)
bottom.set_ylabel('spread, student / teacher')
bottom.set_xscale('log')
bottom.set_xlabel("teacher's principal component (1 = most variance); thin: each, thick: running mean of 9")
plt.show()

blocks = [(1, 8), (9, 32), (33, 128), (129, 512)]
rows = {}
for label in RUNS:
    for a, b in blocks:
        part = slice(a - 1, b)
        rows[(label, f'{a}-{b}')] = {
            'teacher variance share': spread[part].sum() / spread.sum(),
            'ρ² (mean)': rho2[label][part].mean(),
            'spread ratio (mean)': size[label][part].mean(),
            'R² (variance-weighted)': 1 - ((Zt[:, part] - Zs[label][:, part]) ** 2).sum() / spread[part].sum(),
        }
table = pd.DataFrame(rows).T
table.index.names = ['model', 'components']
display(table.round(3))
reproduced = {label: 1 - ((Zt - Zs[label]) ** 2).sum() / spread.sum() for label in RUNS}
print('share of all the teacher\'s variation reproduced (R², all components):',
      ', '.join(f'{label} {value:.3f}' for label, value in reproduced.items()))

**How to read it:** high ρ² with a spread ratio below 1 = the student *knows* the direction but is timid along it
(a calibration problem: part of it could even be fixed by rescaling). Low ρ² = the information is missing, and no
rescaling helps: the student needs better inputs, capacity or a loss that cares about that direction. The teacher
variance share says how much each block matters at all. Compare the two students: does more training data help the
coarse components, the fine ones, or both?

## 4. Relations between tracks

### Similarity matrices

Search and neighbours never look at a vector alone, only at **how vectors relate**. A similarity matrix shows that
directly: cell (i, j) is the centred cosine between track i and track j. Below, 40 test tracks ordered along the
teacher's first axis, teacher left, student right, same colour scale. If the student kept the relations, the two
pictures look alike. (Relational distillation trains the student to match exactly these matrices.)

In [ ]:
pick40 = np.random.default_rng(5).choice(len(t_tracks), 40, replace=False)
pick40 = pick40[np.argsort(zt[pick40, 0])]
ct40, cs40 = unit(t_tracks[pick40] - mu), unit(s_tracks[BASE][pick40] - mu)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8))
for ax, matrix, title in zip(axes, (ct40 @ ct40.T, cs40 @ cs40.T), ('Teacher', 'Student')):
    image = ax.imshow(matrix, cmap=DIVERGING, vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(f'{title}: centred cosine between 40 tracks')
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(image, ax=axes, label='centred cosine', fraction=0.025, pad=0.02)
plt.show()

off_diagonal = ~np.eye(len(t_tracks), dtype=bool)
ct_all = unit(t_tracks - mu)
teacher_sims = (ct_all @ ct_all.T)[off_diagonal]
relation_corr = {}
for label in RUNS:
    cs_all = unit(s_tracks[label] - mu)
    relation_corr[label] = np.corrcoef(teacher_sims, (cs_all @ cs_all.T)[off_diagonal])[0, 1]
    print(f'{label}: correlation of all track-pair similarities with the teacher\'s: {relation_corr[label]:.3f}')

### What do the neighbours need?

The top-10 overlap asks whether a track's 10 nearest neighbours are the same as the teacher's. Here the *teacher* is
reduced to its first k principal components, and the same overlap is measured against the full teacher. That gives
a scale: "neighbours as good as a teacher with k directions". The students' overlaps are drawn on it.

**Built-in check:** at k = 512 nothing is removed, so the overlap must be 1.0, up to float rounding when two
neighbours are almost tied.

In [ ]:
ks = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
overlap_k = [top_k_overlap(track_vectors(mu + Zt[:, :k] @ Vt[:k], test_tracks)[0], t_tracks) for k in ks]
student_overlap = {label: top_k_overlap(s_tracks[label], t_tracks) for label in S}
display(pd.DataFrame({'teacher with k components': overlap_k}, index=pd.Index(ks, name='k')).round(3).T)

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(ks, overlap_k, color=MUTED, marker='o', ms=6, label='teacher, first k components')
for label in RUNS:
    ax.axhline(student_overlap[label], color=MODEL_COLOUR[label], linestyle='--', linewidth=1.5,
               label=f'{label}: {student_overlap[label]:.3f}')
ax.set_xscale('log', base=2)
ax.set_xlabel('teacher principal components kept (k)')
ax.set_ylabel('top-10 overlap with the full teacher')
ax.set_title('How many directions do the neighbours need?')
ax.legend(loc='upper left')
plt.show()

### The overlap, seen

25 random test tracks (rows). In each row, the teacher's 10 nearest tracks, nearest first (columns). Blue = the
student's 10 nearest include it too; grey = the student missed it. The share of blue is the top-10 overlap. Does the
student find the *nearest* ones (left) more often than the 10th?

In [ ]:
def nearest(vectors, k=10):
    sims = vectors @ vectors.T
    np.fill_diagonal(sims, -np.inf)
    return np.argsort(-sims, axis=1)[:, :k]


teacher_nn, student_nn = nearest(t_tracks), nearest(s_tracks[BASE])
found = np.array([[n in set(student_nn[q]) for n in teacher_nn[q]] for q in range(len(t_tracks))])
queries = np.random.default_rng(4).choice(len(t_tracks), 25, replace=False)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.pcolormesh(found[queries], cmap=ListedColormap(['#efefef', SERIES[0]]), edgecolors='white', linewidth=2)
ax.invert_yaxis()
ax.set_xticks(np.arange(10) + 0.5, range(1, 11))
ax.set_yticks([])
ax.grid(False)
ax.set_xlabel("the teacher's neighbours, nearest first")
ax.set_ylabel('25 test tracks')
ax.set_title(f"{found[queries].mean():.0%} of the teacher's 10 nearest are also the student's")
ax.legend(handles=[Patch(color=SERIES[0], label="also in the student's 10"), Patch(color='#efefef', label='missed')],
          loc='upper left', bbox_to_anchor=(1.01, 1))
plt.show()
print('found, by the teacher\'s rank (all test tracks):', ' '.join(f'{v:.0%}' for v in found.mean(0)))

names = [f'{artist or "?"} - {Path(path).stem}' for artist, path in
         zip(test_manifest.loc[track_ids, 'artist'], test_manifest.loc[track_ids, 'file_path'])]
q = queries[0]
print(f'\nexample, query: {names[q]}')
header = "teacher's 10 nearest"
print(f'  {header:60s} student\'s 10 nearest')
for a, b in zip(teacher_nn[q], student_nn[q]):
    print(f'  {"*" if a in set(student_nn[q]) else " "} {names[a][:56]:58s} {"*" if b in set(teacher_nn[q]) else " "} '
          f'{names[b][:56]}')
print('  (* = in both lists)')

## 5. Error analysis: where does the student miss?

Averages hide where a model fails. **Error analysis** is the step that usually pays off most (Andrew Ng spends whole
chapters of *Machine Learning Yearning* on it). The standard moves:

1. **Look at the distribution**, not the mean: is the error spread evenly, or is there a long tail of bad cases?
2. **Slice it:** cut the test set by things you know about each sample and compare the error per slice. Here:
   where in the track the window is, how loud it is, its genre, and how **new** it is (how close the nearest
   *training* window is to it in the teacher's space: far = something the student never saw anything like).
3. **Whole tracks or single windows?** If a few tracks fail entirely, it is about the music; if failures are spread
   over windows of otherwise good tracks, it is about moments (breaks, intros, silence).
4. **Look at the worst cases yourself:** listen to them. The numbers say *where*, the examples say *why*.
5. **Decide:** a slice that matters and fails → targeted data or weighting; a slice that doesn't matter (silence) →
   accept it, or skip such windows in the app; an error that is really the *teacher's* (an unstable target) → not the
   student's fault.

The measure here is the centred cosine per window (1 = perfect, 0 = knows nothing about this window).

In [ ]:
agree = centred[BASE]
frames = test_manifest.loc[test_tracks, 'frames'].to_numpy()
position = np.minimum((window_index / frames * 6).astype(int), 5)   # which sixth of the track (the 6 sampled slots)

centred_train = unit(T_train - mu)
centred_test = unit(T - mu)
novelty = np.concatenate([(centred_test[i:i + 1024] @ centred_train.T).max(1)    # closest training window
                          for i in range(0, len(centred_test), 1024)])

worst = agree <= np.percentile(agree, 5)
print(f'centred cosine: mean {agree.mean():.3f}, median {np.median(agree):.3f}, worst 5% below '
      f'{np.percentile(agree, 5):.3f}')

fig, ax = plt.subplots(figsize=(7.5, 3))
ax.hist(agree, bins=70, color=SERIES[0], alpha=0.85)
ax.axvline(np.percentile(agree, 5), color=TEXT, linewidth=1.2, linestyle='--')
ax.text(np.percentile(agree, 5), ax.get_ylim()[1] * 0.92, '  worst 5%', color=TEXT)
ax.set_xlabel('centred cosine with the teacher')
ax.set_ylabel('windows')
ax.set_title(f'Per-window agreement, {BASE}')
plt.show()

### Slices

Each panel cuts the test windows into ten equal-sized groups by one property and shows the mean agreement per group
(dot) and the middle half of the windows (band). A flat line = this property doesn't matter; a slope = it does.

In [ ]:
def by_bins(ax, values, agreement, colour=SERIES[0], bins=10, discrete=False):
    if discrete:
        groups = np.unique(values)
        members = [values == g for g in groups]
        centres = groups
    else:
        edges = np.quantile(values, np.linspace(0, 1, bins + 1))
        which = np.clip(np.searchsorted(edges, values, side='right') - 1, 0, bins - 1)
        members = [which == b for b in range(bins)]
        centres = [np.median(values[m]) for m in members]
    means = [agreement[m].mean() for m in members]
    low = [np.percentile(agreement[m], 25) for m in members]
    high = [np.percentile(agreement[m], 75) for m in members]
    ax.fill_between(centres, low, high, color=colour, alpha=0.15, linewidth=0)
    ax.plot(centres, means, color=colour, marker='o', ms=6)


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
by_bins(axes[0], position + 1, agree, discrete=True)
axes[0].set_xticks(range(1, 7), ['near\nintro', '2', '3', '4', '5', 'near\noutro'])
axes[0].set_title('Where in the track')
axes[0].set_ylabel('centred cosine')
by_bins(axes[1], loudness, agree)
axes[1].set_xlabel('average log-mel level (quiet → loud)')
axes[1].set_title('How loud the window is')
by_bins(axes[2], novelty, agree)
axes[2].set_xlabel('similarity to the closest training window')
axes[2].set_title('How new it is to the student')
plt.show()

print(f'correlation of agreement with loudness: {np.corrcoef(loudness, agree)[0, 1]:.2f}, '
      f'with closeness to training data: {np.corrcoef(novelty, agree)[0, 1]:.2f}')

### By genre, and: whole tracks or single windows?

In [ ]:
tag_of_track = dict(zip(track_ids, first_tag))
window_tag = np.array([tag_of_track[t] or '(no tag)' for t in test_tracks])
common_tags = [tag for tag, _ in Counter(window_tag).most_common(10)]
per_tag = pd.DataFrame({'windows': [(window_tag == tag).sum() for tag in common_tags],
                        'centred cosine': [agree[window_tag == tag].mean() for tag in common_tags]},
                       index=common_tags).sort_values('centred cosine')

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.barh(per_tag.index, per_tag['centred cosine'], color=SERIES[0], height=0.7)
ax.axvline(agree.mean(), color=TEXT, linewidth=1.2, linestyle='--')
ax.text(agree.mean(), len(per_tag) - 0.3, ' all windows', color=TEXT, va='bottom')
for y, (n, value) in enumerate(zip(per_tag['windows'], per_tag['centred cosine'])):
    ax.text(value, y, f'  {n} windows', va='center', color=MUTED, fontsize=8)
ax.set_xlabel('mean centred cosine')
ax.set_title('Agreement by first genre tag (10 most common)')
ax.grid(axis='y', visible=False)
plt.show()

per_track = pd.Series(agree).groupby(test_tracks).mean()
between = per_track.loc[test_tracks].to_numpy()
share_between = between.var() / agree.var()
print(f'{share_between:.0%} of the variation in agreement is between tracks, {1 - share_between:.0%} within tracks')

**How to read the last line:** if most of the variation is *between* tracks, whole songs are hard (a style the
student handles badly: more such music would help). If most is *within* tracks, it is about moments inside songs
(breaks, quiet parts), and the same song has good and bad windows.

### Where the misses are on the map

The 2D map from section 2, every test track coloured by how badly the student places it (darker = worse). Clusters
of dark dots = a region of the music space the student handles badly.

In [ ]:
miss = 1 - per_track.loc[track_ids].to_numpy()
fig, ax = plt.subplots(figsize=(7.5, 5.5))
order = np.argsort(miss)                                   # worst drawn last, on top
points = ax.scatter(zt[order, 0], zt[order, 1], c=miss[order], cmap=SEQUENTIAL, s=22, edgecolors='white',
                    linewidths=0.5)
fig.colorbar(points, ax=ax, label='miss (1 - centred cosine), track mean', fraction=0.04)
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title("Each test track on the teacher's map, coloured by the student's miss")
plt.show()

### The worst windows: go and listen

The 12 windows the student gets most wrong, with the time inside the track, so you can play them. Ask: is it
silence, a breakdown, speech, an unusual sound? Would a human say the teacher's vector is even meaningful there?

In [ ]:
name_of_track = dict(zip(track_ids, names))
columns = {'track': [name_of_track[t] for t in test_tracks],
           'at': [f'{int(w) * 10 // 60}:{int(w) * 10 % 60:02d}' for w in window_index],
           'genre': window_tag, 'centred cos': agree, 'raw cos': raw[BASE],
           'loudness pct': pd.Series(loudness).rank(pct=True).to_numpy() * 100,
           'closeness pct': pd.Series(novelty).rank(pct=True).to_numpy() * 100}
worst_windows = pd.DataFrame(columns).sort_values('centred cos').head(12)
pd.set_option('display.max_colwidth', 60)
display(worst_windows.round(2).reset_index(drop=True))

`loudness pct` and `closeness pct` are percentiles among all test windows (0 = quietest / least like the training
data). If the worst windows are mostly quiet, or mostly far from the training data, the slices above already said
so; listening tells you whether it matters for a DJ.

Two identical rows in the list mean the **same song twice**: a duplicate file (e.g. one copy in two folders). Error
analysis often finds data problems like this, so check how common they are, whether a duplicate could sit in two
splits (that would leak), and whether the copies flatter a metric.

Found on 2026-11-25: about a fifth of the library's files are copies (one folder copied into another). None leaks
(the only cross-split copies pair the test set with the unused holdout). Copies are identical for the teacher, so a
test track's copy is its nearest neighbour for both models: the "nearest found" share of section 4 is inflated,
while the top-10 overlap moves by under 0.01 when only one copy per song is kept.

In [ ]:
manifest_all = pd.read_parquet(DATA / 'manifest.parquet')
song = (manifest_all.artist.fillna('').str.strip().str.lower() + ' / '
        + manifest_all.file_path.map(lambda p: Path(p).stem.strip().lower()))
copies = manifest_all[song.duplicated(keep=False)].assign(song=song)
splits_per_song = copies.groupby('song').split.nunique()
print(f'{copies.song.nunique()} songs exist more than once ({len(copies)} files) among {len(manifest_all):,} tracks')
print(f'of these, {(splits_per_song > 1).sum()} sit in more than one split (a leak unless holdout is involved)')
print('files per split:', copies.split.value_counts().to_dict())

# Do copies inside the test set flatter the neighbour metrics? A copy is the nearest neighbour of its twin, for the
# teacher and the student alike: an easy hit.
test_song = song.loc[track_ids].to_numpy()
has_copy = pd.Series(test_song).duplicated(keep=False).to_numpy()
one_copy = ~pd.Series(test_song).duplicated(keep='first').to_numpy()
print(f'\ntest tracks with a copy in the test set: {has_copy.sum()} of {len(has_copy)}; teacher cosine between '
      f'copies: min {min(t_tracks[a] @ t_tracks[b] for a, b in [np.flatnonzero(test_song == s)[:2] for s in set(test_song[has_copy])]):.4f}')
print(f'nearest neighbour found by the student: {found[has_copy, 0].mean():.0%} for tracks with a copy, '
      f'{found[~has_copy, 0].mean():.0%} for the others')
print(f'top-10 overlap, {BASE}: all test tracks {student_overlap[BASE]:.3f}, '
      f'one copy per song {top_k_overlap(s_tracks[BASE][one_copy], t_tracks[one_copy]):.3f}')

## 6. One standard number, and a summary

**Linear CKA** (Centred Kernel Alignment) compares two representations of the same samples: 1 = the same geometry
(the same pairwise structure, up to a rotation and a scale), 0 = unrelated. It is the standard tool in the
literature for "do these two networks represent the data alike?"

Note what it ignores: a student rotated away from the teacher can have CKA ≈ 1 and still be useless here, because the
teacher's *text* encoder needs the student's vectors in the teacher's own coordinates. So CKA answers "is the
structure there?", the cosine answers "is it in the right place?". Computed on track vectors.

In [ ]:
def linear_cka(x, y):
    x, y = x - x.mean(0), y - y.mean(0)
    return np.linalg.norm(y.T @ x) ** 2 / (np.linalg.norm(x.T @ x) * np.linalg.norm(y.T @ y))


summary = pd.DataFrame({
    label: {
        'raw cosine (window)': raw[label].mean(),
        'centred cosine (window)': centred[label].mean(),
        'variance reproduced (R², all)': reproduced[label],
        'pair-similarity correlation (track)': relation_corr[label],
        'top-10 overlap (track)': student_overlap[label],
        'linear CKA (track)': linear_cka(s_tracks[label], t_tracks),
    } for label in RUNS}).T
display(summary.round(3))

## What to take from it

- **Raw vs centred cosine:** how much of the headline number is free.
- **R² by component block:** coarse structure kept, fine detail lost? Then losses that work on the *relations*
  between tracks (relational distillation) or on the residual after the mean (centring) target exactly that gap.
- **Neighbours vs k:** how far the students are from the detail the neighbours need.
- **Error analysis:** whether the misses are about *music* the student never saw (→ data), about *moments* such as
  quiet parts (→ maybe irrelevant, or skip them in the app), or spread evenly (→ the model or the loss).
- **CKA vs cosine:** high CKA with a lower cosine would mean the structure is learned but misplaced.

## Exercises

1. Add the 40-epoch run (`'cnn4-small-64-n1199-long'`) or the bigger Phase 1 student (`'cnn4-64-n1500'`) to `RUNS`.
   Which component blocks and which slices gain?
2. Fit the mean and the PCA on the *test* windows instead of the training ones. How much do the numbers move, and
   why is that the wrong way?
3. Set `BASE` to the 50-track student and rerun section 5: does it miss in the same places, only more?
4. Listen to the three worst windows. Write down what they have in common before looking at the slices again.